[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/19_3d_pointcloud_and_detection.ipynb)

# 19. 3D point-cloud and detection ops

set pooling에서 kNN/FPS/EdgeConv/point attention, voxelization, BEV scatter와 center decode까지 진행한다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. PointNet symmetric pooling

point 순서와 무관한 max pooling으로 global feature를 만든다.


In [ ]:
points = torch.tensor(
    [[0.,0.,0.],[1.,0.,0.],[0.,1.,0.],[0.,0.,1.]],
    device=device,
)
mlp = nn.Linear(3, 4).to(device)
feat = F.relu(mlp(points))
global_feat = feat.max(dim=0).values

print(feat)
print("global:", global_feat)


In [ ]:
_ = profile_call("PointNet max pool", lambda p: F.relu(mlp(p)).max(0).values, points)


## 2. kNN

pairwise distance에서 가장 가까운 이웃 index를 고른다.


In [ ]:
dist = torch.cdist(points, points)
knn = dist.topk(k=3, largest=False).indices[:, 1:]
print("distance:\n", dist)
print("2-NN indices:\n", knn)


In [ ]:
_ = profile_call("cdist + topk", lambda p: torch.cdist(p,p).topk(3,largest=False).indices[:,1:], points)


## 3. Farthest point sampling

현재 선택 집합에서 가장 먼 점을 반복 선택한다.


In [ ]:
selected = [0]
min_dist = torch.cdist(points, points[[0]]).squeeze(1)

for _ in range(2):
    idx = int(min_dist.argmax())
    selected.append(idx)
    min_dist = torch.minimum(min_dist, torch.cdist(points, points[[idx]]).squeeze(1))

print("selected:", selected)


## 4. EdgeConv

neighbor-center와 center feature를 concat해 edge feature를 만든다.


In [ ]:
center = points[:, None, :].expand(-1, knn.size(1), -1)
neighbors = points[knn]
edge = torch.cat([center, neighbors - center], dim=-1)

edge_mlp = nn.Linear(6, 4).to(device)
edge_feat = F.relu(edge_mlp(edge)).max(dim=1).values

print("edge feature:", edge_feat.shape)


In [ ]:
_ = profile_call("EdgeConv core", lambda: F.relu(edge_mlp(edge)).max(1).values)


## 5. Point attention

point feature 사이 scaled attention을 적용한다.


In [ ]:
q = feat[None, None]
out = F.scaled_dot_product_attention(q, q, q)
print(out.shape)


In [ ]:
_ = profile_call("point attention", F.scaled_dot_product_attention, q, q, q)


## 6. Voxelization

좌표를 integer voxel index로 양자화한다.


In [ ]:
voxel_size = 0.5
voxel_idx = torch.floor(points / voxel_size).to(torch.int64)
print(voxel_idx)


In [ ]:
_ = profile_call("voxel index", lambda p: torch.floor(p/voxel_size).to(torch.int64), points)


## 7. Pillar / BEV scatter

point feature를 2D grid 위치로 scatter-add한다.


In [ ]:
bev = torch.zeros(1, 4, 4, device=device)
xy = torch.tensor([[0,0],[1,0],[0,1],[1,1]], device=device)
values = torch.tensor([1.,2.,3.,4.], device=device)

bev[0].index_put_((xy[:,1], xy[:,0]), values, accumulate=True)
print(bev)


In [ ]:
def bev_scatter_once():
    grid = torch.zeros(4, 4, device=device)
    return grid.index_put(
        (xy[:, 1], xy[:, 0]),
        values,
        accumulate=True,
    )

_ = profile_call("BEV scatter add", bev_scatter_once)


## 8. Center-based decode

BEV heatmap top-k를 3D center 후보로 바꾼다.


In [ ]:
heat = torch.tensor([[0.1,0.7],[0.2,0.9]], device=device)
score, idx = heat.flatten().topk(2)
y = idx // heat.size(1)
x = idx % heat.size(1)

print("score:", score)
print("center xy:", torch.stack([x,y],-1))


In [ ]:
_ = profile_call("center topk", lambda: heat.flatten().topk(2))


## References and provenance

**[19.1] PointNet / PointNet++**
- 출처: Qi et al.
- 이 노트북에서 가져온 부분: symmetric pooling, FPS/grouping

**[19.2] DGCNN**
- 출처: Wang et al., Dynamic Graph CNN
- 이 노트북에서 가져온 부분: EdgeConv

**[19.3] Point Transformer**
- 출처: Zhao et al., Point Transformer
- 이 노트북에서 가져온 부분: local point attention

**[19.4] PointPillars / CenterPoint**
- 출처: Lang et al.; Yin et al.
- 이 노트북에서 가져온 부분: pillar/BEV scatter and center-based detection

**[19.5] 3DETR / OpenPCDet**
- 출처: Misra et al.; OpenPCDet
- 이 노트북에서 가져온 부분: transformer 3D detection and practical CUDA op references
